SQL 1 — 表樣本（頭 100 行）

In [ ]:
-- fact_sessions 樣本
SELECT * FROM marts.fact_sessions LIMIT 100;

-- dim_customers 樣本
SELECT * FROM marts.dim_customers LIMIT 100;

-- dim_traffic 樣本
SELECT * FROM marts.dim_traffic LIMIT 100;

SQL 2 — 漏斗轉換率（核心分析）

In [ ]:
SELECT
    COUNT(DISTINCT session_id)                                          AS total_sessions,
    SUM(is_session_start)                                               AS session_starts,
    SUM(is_view_item)                                                   AS view_item,
    SUM(is_add_to_cart)                                                 AS add_to_cart,
    SUM(is_begin_checkout)                                              AS begin_checkout,
    SUM(is_purchase)                                                    AS purchases,

    ROUND(SUM(is_view_item)::NUMERIC    / NULLIF(COUNT(DISTINCT session_id), 0) * 100, 2) AS view_rate_pct,
    ROUND(SUM(is_add_to_cart)::NUMERIC  / NULLIF(SUM(is_view_item), 0)        * 100, 2) AS add_to_cart_rate_pct,
    ROUND(SUM(is_begin_checkout)::NUMERIC / NULLIF(SUM(is_add_to_cart), 0)    * 100, 2) AS checkout_rate_pct,
    ROUND(SUM(is_purchase)::NUMERIC     / NULLIF(SUM(is_begin_checkout), 0)   * 100, 2) AS purchase_rate_pct,
    ROUND(SUM(is_purchase)::NUMERIC     / NULLIF(COUNT(DISTINCT session_id), 0) * 100, 2) AS overall_conversion_pct,

    ROUND(SUM(session_revenue)::NUMERIC, 2)                            AS total_revenue
FROM marts.fact_sessions;

SQL 3 — 按 Traffic Source × Device 的漏斗

In [ ]:
SELECT
    t.traffic_source,
    t.traffic_medium,
    f.device_category,
    COUNT(DISTINCT f.session_id)                                              AS total_sessions,
    SUM(f.is_view_item)                                                       AS view_item,
    SUM(f.is_add_to_cart)                                                     AS add_to_cart,
    SUM(f.is_begin_checkout)                                                  AS begin_checkout,
    SUM(f.is_purchase)                                                        AS purchases,
    ROUND(SUM(f.is_purchase)::NUMERIC / NULLIF(COUNT(DISTINCT f.session_id), 0) * 100, 2) AS conversion_rate_pct,
    ROUND(SUM(f.session_revenue)::NUMERIC, 2)                                 AS total_revenue
FROM marts.fact_sessions f
LEFT JOIN marts.dim_traffic t ON f.traffic_sk = t.traffic_sk
GROUP BY t.traffic_source, t.traffic_medium, f.device_category
ORDER BY total_sessions DESC;

4A. 用現有欄位做「國家 × LTV」行為分析

In [ ]:
SELECT
    c.main_country,
    ROUND(AVG(c.lifetime_value)::NUMERIC, 2)                                  AS avg_ltv,
    ROUND(AVG(c.total_orders)::NUMERIC, 2)                                    AS avg_total_orders,
    COUNT(DISTINCT f.session_id)                                              AS total_sessions,
    SUM(f.is_view_item)                                                       AS view_item,
    SUM(f.is_add_to_cart)                                                     AS add_to_cart,
    SUM(f.is_begin_checkout)                                                  AS begin_checkout,
    SUM(f.is_purchase)                                                        AS purchases,
    ROUND(SUM(f.is_purchase)::NUMERIC / NULLIF(COUNT(DISTINCT f.session_id), 0) * 100, 2) AS conversion_rate_pct,
    ROUND(SUM(f.session_revenue)::NUMERIC, 2)                                 AS total_revenue,
    ROUND(SUM(f.session_revenue)::NUMERIC / NULLIF(SUM(f.is_purchase), 0), 2) AS avg_order_value
FROM marts.dim_customers   AS c
LEFT JOIN marts.fact_sessions AS f
       ON c.customer_id = f.customer_id
GROUP BY
    c.main_country
ORDER BY
    total_revenue DESC;

4B：在 SQL 裡用 lifetime_value 做「動態 LTV 分群」

In [ ]:
WITH cust AS (
    SELECT
        customer_id,
        main_country,
        lifetime_value,
        total_orders,
        NTILE(4) OVER (ORDER BY lifetime_value) AS ltv_sort   -- 1~4，4 為最高 LTV
    FROM marts.dim_customers
),
cust_seg AS (
    SELECT
        customer_id,
        main_country,
        lifetime_value,
        total_orders,
        ltv_sort,
        CASE
            WHEN ltv_sort = 4 THEN 'VIP'
            WHEN ltv_sort = 3 THEN 'High'
            WHEN ltv_sort = 2 THEN 'Mid'
            ELSE 'Low'
        END AS ltv_segment
    FROM cust
)
SELECT
    c.ltv_segment,
    c.ltv_sort,
    ROUND(AVG(c.lifetime_value)::NUMERIC, 2)                                  AS avg_ltv,
    ROUND(AVG(c.total_orders)::NUMERIC, 2)                                    AS avg_total_orders,
    COUNT(DISTINCT f.session_id)                                              AS total_sessions,
    SUM(f.is_view_item)                                                       AS view_item,
    SUM(f.is_add_to_cart)                                                     AS add_to_cart,
    SUM(f.is_begin_checkout)                                                  AS begin_checkout,
    SUM(f.is_purchase)                                                        AS purchases,
    ROUND(SUM(f.is_purchase)::NUMERIC / NULLIF(COUNT(DISTINCT f.session_id), 0) * 100, 2) AS conversion_rate_pct,
    ROUND(SUM(f.session_revenue)::NUMERIC, 2)                                 AS total_revenue,
    ROUND(SUM(f.session_revenue)::NUMERIC / NULLIF(SUM(f.is_purchase), 0), 2) AS avg_order_value
FROM cust_seg c
LEFT JOIN marts.fact_sessions f
       ON c.customer_id = f.customer_id
GROUP BY
    c.ltv_segment,
    c.ltv_sort
ORDER BY
    c.ltv_sort;

SQL 5：客戶層級 KPI

In [ ]:
SELECT
    c.customer_id,
    c.main_country,
    c.lifetime_value,
    c.total_orders,
    COUNT(DISTINCT f.session_id)                                              AS total_sessions,
    SUM(f.is_purchase)                                                        AS total_purchases,
    ROUND(SUM(f.session_revenue)::NUMERIC, 2)                                 AS total_revenue,
    ROUND(
        SUM(f.is_purchase)::NUMERIC
        / NULLIF(COUNT(DISTINCT f.session_id), 0) * 100
    , 2)                                                                      AS conversion_rate_pct
FROM marts.dim_customers   AS c
LEFT JOIN marts.fact_sessions AS f
       ON c.customer_id = f.customer_id
GROUP BY
    c.customer_id,
    c.main_country,
    c.lifetime_value,
    c.total_orders
ORDER BY
    c.lifetime_value DESC
LIMIT 200;

SQL 6：Traffic Source × Country × LTV 分佈

In [ ]:
SELECT
    t.traffic_source,
    t.traffic_medium,
    c.main_country,
    COUNT(DISTINCT f.session_id)                    AS sessions,
    SUM(f.is_purchase)                              AS purchases,
    ROUND(SUM(f.session_revenue)::NUMERIC, 2)       AS revenue,
    ROUND(AVG(c.lifetime_value)::NUMERIC, 2)        AS avg_ltv,
    ROUND(AVG(c.total_orders)::NUMERIC, 2)          AS avg_total_orders
FROM marts.fact_sessions f
LEFT JOIN marts.dim_customers c
       ON f.customer_id = c.customer_id
LEFT JOIN marts.dim_traffic  t
       ON f.traffic_sk  = t.traffic_sk
GROUP BY
    t.traffic_source,
    t.traffic_medium,
    c.main_country
ORDER BY
    revenue DESC;